In [1]:
import csv
from pathlib import Path
from rdflib import Namespace, URIRef, BNode,RDF, Literal, ConjunctiveGraph
from rdflib.namespace import RDF, split_uri
from collections import defaultdict
import folium
from shapely import wkt
from shapely.geometry import mapping
from IPython.display import display
import branca.colormap as cm
import pandas as pd
import ipywidgets as widgets
import plotly.express as px

In [2]:
saref = Namespace("https://saref.etsi.org/core/")
s4ener = Namespace("https://saref.etsi.org/saref4ener/")
s4bldg = Namespace("https://saref.etsi.org/saref4bldg/")
geo = Namespace("http://www.opengis.net/ont/geosparql#")
EX = Namespace("https://example.org/")

In [3]:
g = ConjunctiveGraph()

folder = Path("one")

for file in folder.glob("*.log"):
    g.parse(file, format="trig")

C:\Users\gin\AppData\Local\Temp\ipykernel_31464\1050697132.py:1: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = ConjunctiveGraph()


In [4]:
print("Statements in total:", len(g))

Statements in total: 4158741


In [5]:
print("Number of graphs:", len(list(g.contexts())))

Number of graphs: 336998


In [6]:
with open("meter_locations.csv", newline="", encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        meter_uri = row["meter_uri"]
        meter_long = row["meter_long"].strip()
        meter_lat = row["meter_lat"].strip()

        meter = URIRef(meter_uri)
        meter_geom = BNode()

        g.add((meter, RDF.type, saref.Meter))
        g.add((meter, geo.hasGeometry, meter_geom))
        
        if meter_long and meter_lat:
            meter_long = float(meter_long)
            meter_lat = float(meter_lat)

            coord = f"POINT({meter_long} {meter_lat})"
            g.add((meter_geom, geo.asWKT, Literal(coord)))

        bldgs = row["metered_location"].split("/")

        for b in bldgs:
            if b != "":
                bldg = URIRef(EX[b.strip()])
                g.add((meter, EX.measuresBuilding, bldg))

In [7]:
with open("kadaster_buildings.csv", newline="", encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        gebouw = row["gebouw"]
        build = row["build"]

        bldg = EX[build]
        bldg_geom = BNode()

        g.add((bldg, RDF.type, s4bldg.Building))
        g.add((bldg, geo.hasGeometry, bldg_geom))
        g.add((bldg_geom, geo.asWKT, Literal(gebouw)))

In [8]:
# Which buildings does meter X measure?

q = """
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX EX: <https://example.org/>

SELECT ?meter ?building WHERE {
  ?meter a saref:Meter ;
         EX:measuresBuilding ?building .
}
"""

meter_bldgs = defaultdict(list)

for meter_uri, bldg_uri in g.query(q):
    meter_name = meter_uri.split("meter/")[1]
    _, bldg_name = split_uri(bldg_uri)
    
    meter_bldgs[meter_name].append(bldg_name)

dropdown = widgets.Dropdown(options=sorted(meter_bldgs.keys()), description="Meter:")
output = widgets.Output()

def show_bldgs(change):
    with output:
        output.clear_output()
        meter = change["new"]
        print(f"Buildings measured by {meter}:")
        for bldg in meter_bldgs[meter]:
            print("-", bldg)

dropdown.observe(show_bldgs, names="value")
display(dropdown, output)
show_bldgs({"new": dropdown.value})

Buildings measured by scu200/SCU-200-B09/31:
- B09


In [9]:
# Is there a meter that does not measure any building?

q = """
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX EX: <https://example.org/>

SELECT ?meter WHERE {
  ?meter a saref:Meter .
  FILTER NOT EXISTS {
    ?meter EX:measuresBuilding ?building .
  }
}
"""

m = folium.Map(location=[51.985213, 5.870865], zoom_start=17)
meters = []

for (meter_uri,) in g.query(q):
    meter_name = meter_uri.split("meter/")[1]
    
    q2 = f"""
    PREFIX geo: <http://www.opengis.net/ont/geosparql#>

    SELECT ?point WHERE{{
      <{meter_uri}> geo:hasGeometry ?geometry.
      ?geometry geo:asWKT ?point.
    }}
    """

    meters.append(meter_name)

    result = g.query(q2)

    if result:
        for (location,) in result:
            coords = wkt.loads(location)
              
            folium.GeoJson(data=mapping(coords), tooltip=meter_name).add_to(m)

print("Meters that do not measure building usage:")
print(", ".join(meters), "\n")
print("Map with such meters that have a known location:")
m

Meters that do not measure building usage:
BCU105420, BCU105405, BCU105369, BCU109794, BCU105423, BCU105418, BCU105425, BCU105404, tap/tap-kb/B12-cp01, tap/tap-kb/B12-cp02, tap/tap-kb/B12-cp03, tap/tap-kb/B12-cp04, 871687120000000080/6500036557, 871687120000092214/6500071036, 871687110003209825, 3266975, netx/bms-opc-ua/IPS-S3.1.1-IP-interface-DIN-railapparaat, netx/bms-opc-ua/FBXi-8R8-331123, netx/bms-opc-ua/AC-S1.2.1-Application-Controller 

Map with such meters that have a known location:


In [10]:
# Is there any building that has no meter measuring its usage?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>
PREFIX EX: <https://example.org/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
  FILTER NOT EXISTS {
    ?meter EX:measuresBuilding ?building
  }

}
"""

m = folium.Map(location=[51.985213, 5.870865], zoom_start=17)
bldgs = []

for (bldg_uri,) in g.query(q):
    _, bldg_name = split_uri(bldg_uri)
    
    q2 = f"""
    PREFIX geo: <http://www.opengis.net/ont/geosparql#>

    SELECT ?polygon WHERE{{
      <{bldg_uri}> geo:hasGeometry ?geometry.
      ?geometry geo:asWKT ?polygon.
    }}
    """

    bldgs.append(bldg_name)

    result = g.query(q2)

    for (poly,) in result:
        polygon = wkt.loads(poly)
              
        folium.GeoJson(data=mapping(polygon), tooltip=bldg_name).add_to(m)

print("Buildings that have no meter measuring their usage:")
print(", ".join(bldgs), "\n")
print("Map with such buildings:")
m

Buildings that have no meter measuring their usage:
B50, B52, B04, B12, B13, B16, B17, B30, B44, B14, B42, M01 

Map with such buildings:


In [11]:
# Which meters measure the same building?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX EX: <https://example.org/>

    SELECT ?meter WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
    }}
    """

    if len(g.query(q2)) > 1:
      _, bldg_name = split_uri(bldg_uri)
      print("Measuring building",bldg_name,":")

      for (meter_uri,) in g.query(q2):
        meter_name = meter_uri.split("meter/")[1]

        print(meter_name)

Measuring building B09 :
scu200/SCU-200-B09/31
scu200/SCU-200-B09/35
Measuring building B20 :
scu200/SCU-200-B31-2/46
scu200/SCU-200-B31-2/47
Measuring building B31 :
scu200/SCU-200-B31-2/45
scu200/SCU-200-B31-1/50
scu200/SCU-200-B31-1/52
Measuring building B46 :
scu200/SCU-200-B46/42
scu200/SCU-200-B46/43
scu200/SCU-200-B46/44


In [12]:
# What is the aggregated daily energy usage of building X?

q = """
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX s4ener: <https://saref.etsi.org/saref4ener/>
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX EX: <https://example.org/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

bldg_kwh = defaultdict()

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX EX: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
      FILTER NOT EXISTS{{
        ?meter a saref4ab:ChargePointMeter
      }}
      FILTER STRSTARTS(
        STR(?meter),
        "https://cgi.com/hedgeiot/meter/"
      )
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/KiloW-HR> .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        bldg_usage = 0
        meter_values = defaultdict(lambda: defaultdict(list))

        for meter_uri, time, value in result:
            minute = time[:16]
            meter_values[meter_uri][minute].append(float(value))

        complete = defaultdict(float)
        for meter_uri, times in meter_values.items():
            for time, values in times.items():
                if len(values) == 4:
                    complete[time] = max(values)
        
        sorted_time = sorted(complete.keys())
        
        earliest = sorted_time[0]
        latest = sorted_time[-1]

        bldg_usage += complete[latest] - complete[earliest]
                
        _, bldg_name = split_uri(bldg_uri)

        bldg_kwh[bldg_name] = bldg_usage

kwh_dropdown = widgets.Dropdown(options=sorted(bldg_kwh.keys()), description="Building:")
kwh_output = widgets.Output()

def show_kwh(change):
    with kwh_output:  
        kwh_output.clear_output()
        bldg = change["new"]
        print(f"Daily aggregated energy usage of {bldg}:")
        print("-", bldg_kwh[bldg], "kWh")

kwh_dropdown.observe(show_kwh, names="value")
display(kwh_dropdown, kwh_output)
show_kwh({"new": kwh_dropdown.value})

Daily aggregated energy usage of B01:
- 476.1699999999837 kWh


In [13]:
# Which building shows the highest average hourly power consumption?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""
max_avg = ("", "", -1)

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX EX: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
      FILTER NOT EXISTS{{
        ?meter a saref4ab:ChargePointMeter
      }}
      FILTER STRSTARTS(
        STR(?meter),
        "https://cgi.com/hedgeiot/meter/"
      )
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/W> .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        _, bldg_name = split_uri(bldg_uri)

        meter_info = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
        for meter_uri, time, value in result:
            hour = time[:13]
            minute = time[:16]
            meter_info[meter_uri][hour][minute].append(float(value))
        
        hourly = defaultdict(list)

        for meter_uri, hours in meter_info.items():
            for hour, minutes in hours.items():
                for minute, values in minutes.items():
                    if len(values) == 4:
                        hourly[hour].append(max(values))

        avge = {}
        for hour in hourly:
            avge[hour] = sum(hourly[hour]) / len(hourly[hour])
            
        for hour, avge in avge.items():
            if avge > max_avg[2]:
                max_avg = (bldg_name, hour, avge)

n, h, a = max_avg
print(f"The building with the highest average hourly power consumption is {n}, with a peak average of {a} W and reaching that during {h[11:]}:00.")

The building with the highest average hourly power consumption is B01, with a peak average of 45435.63892857143 W and reaching that during 08:00.


In [14]:
# How do the daily power usage timelines of buildings X and Y differ?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

kwh_row = []
w_row = []

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX EX: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value ?property WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
      FILTER NOT EXISTS{{
        ?meter a saref4ab:ChargePointMeter
      }}
      FILTER STRSTARTS(
        STR(?meter),
        "https://cgi.com/hedgeiot/meter/"
      )
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn ?property .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        _, bldg_name = split_uri(bldg_uri)
        bldg_kWh = defaultdict(float)
        bldg_W = defaultdict(list)

        meter_vals = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
        for meter_uri, time, value, prpt_uri in result:
            minute = time[:16]
            meter_vals[meter_uri][prpt_uri][minute].append(float(value))

        complete = defaultdict(list)
        meter_counter = 0
        for meter_uri, properties in meter_vals.items():
            meter_counter += 1
            for prpt_uri, times in properties.items():
                prpt_name = prpt_uri.split("/")[-1]
                for time, values in times.items():
                    if prpt_name == "KiloW-HR":
                        if len(values) == 4:
                            complete[time].append(max(values))

                    elif prpt_name == "W":
                        if len(values) == 4:
                            bldg_W[time].append(max(values))
        
        sorted_time = sorted(complete.keys())
        
        for i in range(1, len(sorted_time)):
            prev_t = sorted_time[i-1]
            cur_t = sorted_time[i]

            if len(complete[cur_t]) == meter_counter and len(complete[prev_t]) == meter_counter:
                bldg_kWh[cur_t] += sum(complete[cur_t]) - sum(complete[prev_t])

        for time, val in bldg_kWh.items():
            kwh_row.append({"building": bldg_name, "time": pd.to_datetime(time), "kwh": val})

        for time, vals in bldg_W.items():
            w_row.append({"building": bldg_name, "time": pd.to_datetime(time), "power": sum(vals)})

        if bldg_name == "B09":
            print(bldg_name, sorted(list(complete.items()))[-30:])

kwh_df = pd.DataFrame(kwh_row).sort_values("time")
w_df = pd.DataFrame(w_row).sort_values("time")

bldgs = sorted(w_df["building"].unique())

bldg_x = widgets.Dropdown(options=bldgs, description="Building X:")
bldg_y = widgets.Dropdown(options=bldgs, description="Building Y:")

def show_graphs(building_x, building_y):
    w_filtered = w_df[w_df["building"].isin([building_x, building_y])]
    kwh_filtered = kwh_df[kwh_df["building"].isin([building_x, building_y])]

    fig_w = px.line(
        w_filtered,
        x="time",
        y="power",
        color="building",
        title="Power usage timeline",
        labels={"time": "Time", "power": "Power (W)", "building": "Building"}
    )
    fig_w.show()

    fig_kwh = px.line(
        kwh_filtered,
        x="time",
        y="kwh",
        color="building",
        title="Energy usage timeline",
        labels={"time": "Time", "kwh": "Energy (kWh)", "building": "Building"}
    )
    fig_kwh.show()

widgets.interactive(show_graphs, building_x=bldg_x, building_y=bldg_y)

interactive(children=(Dropdown(description='Building X:', options=('B01', 'B02', 'B09', 'B11', 'B18', 'B19', '…

In [15]:
# Can buildings be visualised spatially, with colours indicating aggregated daily energy usage?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
SELECT ?building ?polygon WHERE {
  ?building a s4bldg:Building ;
            geo:hasGeometry ?geometry .
  ?geometry geo:asWKT ?polygon .
}
"""

m = folium.Map(location=[51.985213, 5.870865], zoom_start=17)
bldg_data = defaultdict(tuple)

for bldg_uri, polygon in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX EX: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
      FILTER NOT EXISTS{{
        ?meter a saref4ab:ChargePointMeter
      }}
      FILTER STRSTARTS(
        STR(?meter),
        "https://cgi.com/hedgeiot/meter/"
      )
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/KiloW-HR> .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)
    bldg_usage = -1

    if result:
        bldg_usage =0
        meter_values = defaultdict(lambda: defaultdict(list))

        for meter_uri, time, value in result:
            minute = time[:16]
            meter_values[meter_uri][minute].append(float(value))

        complete = defaultdict(float)
        for meter_uri, times in meter_values.items():
            for time, values in times.items():
                if len(values) == 4:
                    complete[time] = max(values)
        
        sorted_time = sorted(complete.keys())
        
        earliest = sorted_time[0]
        latest = sorted_time[-1]

        bldg_usage += complete[latest] - complete[earliest]
                
        _, bldg_name = split_uri(bldg_uri)

        bldg_kwh[bldg_name] = bldg_usage
                
    geom = wkt.loads(polygon)
    _, bldg_name = split_uri(bldg_uri)

    bldg_data[bldg_name] = (geom, bldg_usage)

colour_map = cm.LinearColormap(colors=["#ffffb2", "#fd8d3c", "#bd0026"], vmin=0, vmax=max(usage for _, usage in bldg_data.values()))

for name, (location, numbers) in bldg_data.items():
    if numbers == -1:
        colour = "#cccccc"
        message = f"{name} has no meters."
    else:
        colour = colour_map(numbers)
        message = f"{name}: {numbers} kWh"

    folium.GeoJson(data=mapping(location), style_function=lambda feature, color=colour: {"fillColor":color, "color": "black", "weight": 1, "fillOpacity": 0.7,}, tooltip=message).add_to(m)

colour_map.caption = "Aggregated energy usage per day in kWh"
colour_map.add_to(m)

print("Map with spacial representation of the buildings in the business park, colours representing the aggregated energy usage per building:")
m

Map with spacial representation of the buildings in the business park, colours representing the aggregated energy usage per building:


In [ ]:
g.serialize(destination="enriched_ontology.log", format="trig")